In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler,MinMaxScaler,RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PowerTransformer
import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt
from keras.models import Sequential
from keras.layers import Dense,Dropout,BatchNormalization, Input
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping
from keras.metrics import AUC
from imblearn.over_sampling import SMOTE
from scipy.stats import rankdata
import warnings

/home/UR/coelhrod/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
warnings.simplefilter('ignore')
train = pd.read_csv("/kaggle/input/playground-series-s5e3/train.csv", index_col='id')
test = pd.read_csv("/kaggle/input/playground-series-s5e3/test.csv", index_col='id')
train_extra=pd.read_csv("/kaggle/input/rainfall-prediction-using-machine-learning/Rainfall.csv")

In [ ]:
train_extra.columns = train_extra.columns.str.replace(' ', '')
train_extra = train_extra[train_extra.columns].copy()
train_extra['rainfall'] = train_extra['rainfall'].map({'no': 0, 'yes': 1})
train_extra['humidity']=train_extra['humidity'].astype(float)
train_extra['cloud']=train_extra['cloud'].astype(float)
train_features=list(train)
train_extra=train_extra[train_features]

train = pd.concat([train, train_extra], axis=0, ignore_index=True)
train = train.drop_duplicates()
train.shape

In [ ]:
test['winddirection']=test['winddirection'].fillna(value=test['winddirection'].mean())
train['winddirection']=train['winddirection'].fillna(value=train['winddirection'].mean())
train['windspeed']=train['windspeed'].fillna(value=train['windspeed'].mean()) 

## Preprocessing

In [ ]:
def handle_outliers(df, columns):
    clipped_count = 0
    
    # Loop through each specified column
    for col in columns:
        # Calculate Q1, Q3, and IQR for the current column
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        # Define the lower and upper bounds
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        # Count outliers
        outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
        clipped_count += outliers
        
        # Clip the outliers to the bounds
        df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)

    print("Number of outliers clipped: ", clipped_count)
    return df

def scale_data(df,columns_to_scale,scaler="robust"):
    if(scaler=="standard"):
        scaler=StandardScaler()
    elif(scaler=="minmax"):
        scaler=MinMaxScaler()
    elif(scaler=="robust"):
        scaler=RobustScaler()
        
    remaining_columns = df.columns.difference(columns_to_scale)
    
    # Scale the selected columns
    scaler = RobustScaler()
    df_scaled = pd.DataFrame(scaler.fit_transform(df[columns_to_scale]), columns=columns_to_scale)
    
    # Combine scaled and unscaled columns
    df_final = pd.concat([df_scaled, df[remaining_columns].reset_index(drop=True)], axis=1)
    
    return df_final

def transformation(df,feature):
    transformer = PowerTransformer(method='yeo-johnson')  # Initialize Yeo-Johnson transformer
    df[feature] = transformer.fit_transform(df[[feature]])

    return df

def extract_features_from_day(df, day_column):
    # 1. Day of the week (approximate, assuming year starts on a Monday)
    df['day_sin'] = np.sin(2 * np.pi * df['day'] / 365)
    df['day_cos'] = np.cos(2 * np.pi * df['day'] / 365)

    df['day_of_week'] = ((df[day_column] - 1) % 7) + 1

    # 2. Week number
    df['week_number'] = ((df[day_column] - 1) // 7) + 1
    
    
    # 3. Season
    def get_season(day):
        if day <= 79 or day >= 355:
            return 'winter'
        elif day <= 171:
            return 'spring'
        elif day <= 264:
            return 'summer'
        else:
            return 'fall'

    df['season'] = df[day_column].apply(get_season)

    # 4. Half of the year
    df['half_of_year'] = df[day_column].apply(lambda d: 1 if d <= 182 else 2)

    
    window_size = 7
    df['temp_roll_mean'] = df['temparature'].rolling(window=window_size, min_periods=1).mean()
    df['temp_roll_std'] = df['temparature'].rolling(window=window_size, min_periods=1).std()

    df['dew_roll_mean'] = df['dewpoint'].rolling(window=window_size, min_periods=1).mean()
    df['dew_roll_std'] = df['dewpoint'].rolling(window=window_size, min_periods=1).std()

    df['cloud_roll_mean'] = df['cloud'].rolling(window=window_size, min_periods=1).mean()
    df['cloud_roll_std'] = df['cloud'].rolling(window=window_size, min_periods=1).std()
    
    df['press_roll_mean'] = df['pressure'].rolling(window=window_size, min_periods=1).mean()
    df['press_roll_std'] = df['pressure'].rolling(window=window_size, min_periods=1).std()

    df['average_temp']=(df['mintemp']+df['maxtemp'])/2

    df['temp_diff'] = df['temparature'].diff().fillna(0)

    df['pressure_diff']=df['pressure'].diff().fillna(0)

    df['dewpoint_diff']=df['dewpoint'].diff().fillna(0)

    df['average_temp_diff']=df['average_temp'].diff().fillna(0)

    # One-hot encode the season
    season_dummies = pd.get_dummies(df['season'], prefix='season').astype(float)

    # Combine the original DataFrame with the one-hot encoded season columns
    df = pd.concat([df, season_dummies], axis=1).drop("season",axis=1)

    return df

def feature_selection(df):
    
    pca=PCA(n_components=0.99)
    reduced=pca.fit_transform(df)
    n_components=reduced.shape[1]
    pca_columns = [f'PC{i+1}' for i in range(n_components)]
    reduced = pd.DataFrame(reduced, columns=pca_columns)
    print('Number of reduced features: ',df.shape[1]-n_components)
    return reduced,pca

def impute(df):
    # Separate categorical and numerical columns
    try:
        categorical_cols = df.select_dtypes(include=['object']).columns
        cat_imputer = SimpleImputer(strategy='most_frequent')
        df[categorical_cols] = cat_imputer.fit_transform(df[categorical_cols])
    except:
        print("Error in Categorical imputing")
        pass 
    try:
        numerical_cols = df.select_dtypes(exclude=['object']).columns
    # Impute numerical features with the mean value
        num_imputer = SimpleImputer(strategy='mean')
        df[numerical_cols] = num_imputer.fit_transform(df[numerical_cols])
    except:
        print("Error in numerical imputing")
        pass
    
    return df

In [ ]:
def preprocessing_pipeline(X_train,X_test,handle_outliers=False):
    print("train set shape before preprocessing: ",train.shape)
    print("test set shape before preprocessing: ",test.shape)
    target_column='rainfall'
    #X_test=test.drop('id',axis=1)
    #X_train=train.drop('id',axis=1)
    X_train=X_train.drop(target_column,axis=1)
    
    y_train=train[target_column]
    X_train=X_train.drop_duplicates()
    
    X_train=extract_features_from_day(X_train, "day")
    X_test=extract_features_from_day(X_test, "day")

    X_train=impute(X_train)
    X_test=impute(X_test)            

    if(handle_outliers==True):
        X_train=handle_outliers(X_train,X_train.columns)
    
    not_to_scale=['season_fall','season_spring','season_summer','season_winter']
    to_scale = X_train.columns.difference(not_to_scale)
   
    X_train=scale_data(X_train,to_scale)
    X_test=scale_data(X_test,to_scale)
    print("train shape after preprocessing: ",X_train.shape)
    print("test shape after Preprocessing: ",X_test.shape)


    return X_train,y_train,X_test

In [ ]:
X,y,test_set=preprocessing_pipeline(train,test)

In [ ]:
test_set.sample(5)

In [ ]:
smote=SMOTE(sampling_strategy='minority')
X_resample,y_resample=smote.fit_resample(X,y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_resample, y_resample, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

## MLP

In [ ]:
def build_model(hp):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(X_train.shape[1],)),
        tf.keras.layers.Dense(hp.Int('units_1', min_value=128, max_value=512, step=128), activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(hp.Float('dropout_1', 0.2, 0.5, step=0.1)),
        
        tf.keras.layers.Dense(hp.Int('units_2', min_value=64, max_value=256, step=64), activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(hp.Float('dropout_2', 0.2, 0.5, step=0.1)),
        
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(hp.Choice('learning_rate', [0.01, 0.001, 0.0001])),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Criar tuner
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',  # Otimizando precisão de validação
    max_trials=10,  # Número máximo de testes
    executions_per_trial=2,  # Média de execuções para cada combinação
    directory='keras_tuner_dir',  # Diretório para salvar resultados
    project_name='mlp_tuning'
)

# Iniciar busca
tuner.search(X_train, y_train, epochs=50, validation_data=(X_val, y_val))

# Melhor modelo encontrado
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print("Melhores hiperparâmetros:", best_hps.values) 

In [ ]:
# Early stopping to prevent overfitting
early_stopping = EarlyStopping(monitor='val_loss', patience=50, restore_best_weights=True)

# Define the model dynamically
model = Sequential([
    Input(shape=(X_resample.shape[1],)),
    Dense(best_hps['units_1'], activation='relu', kernel_initializer='he_normal'),
    BatchNormalization(),
    Dropout(best_hps['dropout_1']),
    
    Dense(best_hps['units_2'], activation='relu', kernel_initializer='he_normal'),
    BatchNormalization(),
    Dropout(best_hps['dropout_2']),
    
    Dense(64, activation='relu', kernel_initializer='he_normal'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(32, activation='relu', kernel_initializer='he_normal'),
    BatchNormalization(),
    Dense(16, activation='relu', kernel_initializer='he_normal'),
    BatchNormalization(),
    Dense(1, activation='sigmoid')  # Binary classification
])

# Compile the model using the best learning rate
optimizer = Adam(learning_rate=best_hps['learning_rate'])
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=[AUC(name='auc')])

# Train the model
history = model.fit(X_train, y_train, epochs=300, batch_size=32, validation_data=(X_val, y_val), 
                    callbacks=[early_stopping], verbose=1)


In [ ]:
test_loss, test_accuracy = model.evaluate(X_test,y_test)
print(f"Test Loss : {test_loss}")
print(f"Test auc: {test_accuracy}")

In [ ]:
predictions = model.predict(test_set)
result_df = pd.DataFrame({
    "id": np.arange(2190, 2190 + len(predictions)),  # Gera IDs no tamanho correto
    "rainfall": np.squeeze(predictions)  # Remove dimensões extras se necessário
})

print(result_df.head())


## Submission CSV Ensemble

In [ ]:
print("Best Public Notebook achieves LB = 0.954")
best_public = pd.read_csv("/kaggle/input/lb-915-public-notebook/submission95427.csv")
display( best_public.head() )
best_public = best_public.rainfall.values

In [ ]:
sub = pd.read_csv("/kaggle/input/playground-series-s5e3/sample_submission.csv")
sub.rainfall = -0.5 * rankdata( result_df["rainfall"] ) + 1.5 * rankdata( best_public )
sub.rainfall = rankdata( sub.rainfall ) / len(sub)
print( sub.shape )
sub.to_csv(f"submission.csv",index=False)
sub.head()